In [1]:
import andes
from andes.interop.pandapower import to_pandapower
import pandapower as pp
import random
import numpy as np
import re
import pandas as pd


In [2]:
random.seed(42)

TYPE_MAP = {
    'B': 'biomass', 
    'G': 'gas', 
    'C': 'coal', 
    'H': 'hydro',
    'E': 'geothermal', 
    'N': 'nuclear', 
    'SC': 'synchronous condenser',
    'P': 'pump hydro storage', 
    'R': 'generic renewable', 
    'S': 'solar', 
    'W': 'wind',
    'D': 'diesel'
}

# 1. 生成合理的 24小时 测试曲线 (Dummy Curves)
# ==========================================
# 背景负荷乘数曲线：模拟典型的城市用电。深夜用电低谷(0.6)，白天工作时间升高，傍晚达到晚高峰(1.0)
dummy_load_curve = [
    0.65, 0.60, 0.55, 0.55, 0.60, 0.65, 0.75, 0.85, 0.90, 0.95, 0.95, 0.90, 
    0.90, 0.95, 0.95, 0.90, 0.95, 1.00, 1.00, 0.95, 0.90, 0.85, 0.75, 0.70
]

# 风电出力曲线 (标幺值 0~1)：风电通常具有“夜间大、白天小”的逆负荷特性
dummy_wind_curve = [
    0.90, 0.95, 1.00, 0.95, 0.90, 0.80, 0.60, 0.40, 0.30, 0.20, 0.15, 0.15, 
    0.10, 0.15, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.85, 0.90, 0.95
]

# 太阳能出力曲线 (标幺值 0~1)：典型的日照曲线，夜间为0，正午达到峰值
dummy_solar_curve = [
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.10, 0.30, 0.60, 0.80, 0.95, 1.00, 
    1.00, 0.95, 0.85, 0.60, 0.30, 0.10, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00
]

In [3]:
raw_file_path = "case_datasets/WECC_240_bus/240busWECC_2018_PSS.raw"
dyr_file_path = "case_datasets/WECC_240_bus/240busWECC_2018_PSS.dyr"

system = andes.load(raw_file_path, setup=False, no_output=True, default_config=True)
system.setup()
net = to_pandapower(system)
net.bus['bus_id'] = system.Bus.idx.v
net.bus['max_vm_pu'] = 1.5

bus_machines_dict = {}
pattern = re.compile(r"^\s*(\d+)\s+'[^']+'\s+([A-Za-z0-9]+)")

with open(dyr_file_path, 'r') as file:
    for line in file:
        match = pattern.search(line)
        if match:
            bus_id = int(match.group(1))
            machine_id = match.group(2).strip()
            
            if machine_id == 'SC':
                fuel_code = 'SC'
            elif len(machine_id) == 2 and machine_id.isalpha():
                fuel_code = machine_id[1]
            elif len(machine_id) == 1:
                fuel_code = machine_id
            else:
                fuel_code = ''.join([char for char in machine_id if char.isalpha()])
                fuel_code = fuel_code[-1] if fuel_code else machine_id
                
            full_type = TYPE_MAP.get(fuel_code, machine_id)
            
            if bus_id not in bus_machines_dict:
                bus_machines_dict[bus_id] = {}
            bus_machines_dict[bus_id][machine_id] = full_type

net.gen['type'] = 'unknown'

for pp_bus_idx in net.gen['bus'].unique():
    
    original_bus_id = int(net.bus.loc[pp_bus_idx, 'bus_id']) 
    
    if original_bus_id in bus_machines_dict:
        types_available = list(bus_machines_dict[original_bus_id].values())
        gen_indices = net.gen[net.gen['bus'] == pp_bus_idx].index
        for i, gen_idx in enumerate(gen_indices):
            if i < len(types_available):
                net.gen.loc[gen_idx, 'type'] = types_available[i]

  6 slack generators are enabled on 1 bus(es): [3933].
/mnt/miniconda3/envs/migration/lib/python3.12/site-packages/andes/interop/pandapower.py:231: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  bus_df['type'] = 'b'
/mnt/miniconda3/envs/migration/lib/python3.12/site-packages/andes/interop/pandapower.py:232: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  bus_df['in_service'] = bus_df['in_service'].astype('bool')
/mnt/miniconda3/envs/migration/lib/python3.12/site-packages/andes/interop/pandapower.py:256: Settin

In [4]:
net.bus.head(10)

,vn_kv,name,in_service,max_vm_pu,min_vm_pu,zone,type,bus_id
0,500.0,FOURCORN,True,1.5,0.9,3,b,1001
1,345.0,FOURCORN 1,True,1.5,0.9,3,b,1002
2,230.0,FOURCORN 2,True,1.5,0.9,10,b,1003
3,345.0,SAN JUAN,True,1.5,0.9,10,b,1004
4,20.0,FCNGN4CC,True,1.5,0.9,10,b,1032
5,20.0,SJUAN G4,True,1.5,0.9,10,b,1034
6,500.0,CORONADO,True,1.5,0.9,3,b,1101
7,345.0,CHOLLA,True,1.5,0.9,3,b,1102
8,20.0,CORONADO 8,True,1.5,0.9,3,b,1131
9,500.0,MOENKOPI,True,1.5,0.9,3,b,1201


In [5]:
pp.runpp(net)

In [6]:
def setup_datacenters(net, dc_penetration_ratio=0.1, dc_max_mw=200.0):
    total_grid_load_mw = net.load['p_mw'].sum()
    target_dc_load_mw  = total_grid_load_mw * dc_penetration_ratio
    num_dcs = max(1, int(np.ceil(target_dc_load_mw / dc_max_mw)))
    print(f"Total Grid Load: {total_grid_load_mw:.2f} MW, Target DC Load: {target_dc_load_mw:.2f} MW, Number of DCs: {num_dcs}")
    
    available_buses = list(net.bus.index)
    selected_buses = random.sample(available_buses, num_dcs)
    
    dc_load_indices = []
    remaining_dc_load_mw = target_dc_load_mw
    for i, bus_id in enumerate(selected_buses):
        dc_p_mw = min(dc_max_mw, remaining_dc_load_mw)
        dc_idx = pp.create_load(net, bus=bus_id, p_mw=dc_p_mw, q_mvar=0.0, name=f"Datacenter_{i}", controllable=False)
        dc_load_indices.append(dc_idx)
        remaining_dc_load_mw -= dc_p_mw
    
    return net, dc_load_indices

In [7]:
def scale_renewables(net, wind_scale=2.0, solar_scale=2.0):
    wind_mask = net.gen['type'] == 'wind'
    net.gen.loc[wind_mask, 'max_p_mw'] *= wind_scale
    net.gen.loc[wind_mask, 'p_mw'] *= wind_scale
    
    solar_mask = net.gen['type'] == 'solar'
    net.gen.loc[solar_mask, 'max_p_mw'] *= solar_scale
    net.gen.loc[solar_mask, 'p_mw'] *= solar_scale
    
    return net


In [8]:
def apply_time_step(net, hour, load_curve, wind_curve, solar_curve, initial_load, initial_gen_max_p):
    base_load_mask = ~net.load['name'].astype(str).str.startswith("Datacenter")
    current_load_multiplier = load_curve[hour]
    net.load.loc[base_load_mask, 'p_mw'] = initial_load[base_load_mask] * current_load_multiplier
    
    wind_mask = net.gen['type'] == 'wind'
    net.gen.loc[wind_mask, 'max_p_mw'] = initial_gen_max_p[wind_mask] * wind_curve[hour]
    
    solar_mask = net.gen['type'] == 'solar'
    net.gen.loc[solar_mask, 'max_p_mw'] = initial_gen_max_p[solar_mask] * solar_curve[hour]
    
    return net

In [9]:
def setup_generation_costs(net):
    for idx, row in net.gen.iterrows():
        fuel = row.get('type', 'unknown')
        cost_price = 40.0 # 默认价格
        if fuel in ['wind', 'solar']: cost_price = 0.0
        elif fuel == 'hydro': cost_price = 5.0
        elif fuel == 'coal': cost_price = 30.0
        elif fuel == 'gas': cost_price = 50.0
        elif fuel == 'nuclear': cost_price = 30.0
        
        pp.create_poly_cost(net, element=idx, et='gen', cp1_eur_per_mw=cost_price)
    
    if len(net.ext_grid) > 0:
        # 给平衡节点极其宽裕的上下限
        net.ext_grid['min_p_mw'] = -99999.0
        net.ext_grid['max_p_mw'] = 99999.0
        for idx, row in net.ext_grid.iterrows():
            # 假设平衡节点的兜底电价较贵
            pp.create_poly_cost(net, element=idx, et='ext_grid', cp1_eur_per_mw=60.0)
        
    return net

net = setup_generation_costs(net)

In [10]:
def run_simulation(net):
    wind_scale = 2
    solar_scale = 2
    
    dc_ratio = 0.0
    dc_max_mw = 200.0
    
    net = scale_renewables(net, wind_scale=wind_scale, solar_scale=solar_scale)
    
    net, dc_indices = setup_datacenters(net, dc_penetration_ratio=dc_ratio, dc_max_mw=dc_max_mw)
    
    initial_load_p = net.load['p_mw'].copy()
    initial_gen_max_p = net.gen['max_p_mw'].copy()
    
    for hour in range(24):
        apply_time_step(net, hour, dummy_load_curve, dummy_wind_curve, dummy_solar_curve, initial_load_p, initial_gen_max_p)
        
        # Add datacenter workload shifting here
        
        try:
            pp.rundcopp(net)
            overloaded = net.res_line[net.res_line.loading_percent > 99.9]
            total_load_mw = net.load['p_mw'].sum()
            total_gen_dispatch_mw = net.res_gen['p_mw'].sum() if len(net.res_gen) > 0 else 0.0
            total_cost = net.res_cost
            print(f"时间 {hour:02d}:00")
            if len(overloaded) > 0:
                print("    " + overloaded[['p_from_mw', 'p_to_mw', 'loading_percent']].head().to_string(index=False))
            print(f"    总负荷:", total_load_mw)
            print(f"    机组出力:", total_gen_dispatch_mw)
            print(f"    总成本:", total_cost)
            print(f"    绑定线路数:", len(overloaded))
            print(f"    线路最高负载率: {net.res_line['loading_percent'].max():.2f}%")
            if len(net.ext_grid):
                print("    " + net.res_ext_grid[['p_mw']].to_string(index=False))
        except Exception as e:
            print(f"时间 {hour:02d}:00 | OPF 计算失败: {e}")
            print(f"    总负荷:", net.load.p_mw.sum())
            print(f"    机组出力:", net.gen.max_p_mw.sum())
    return
        
    

In [11]:
run_simulation(net)

Total Grid Load: 138806.68 MW, Target DC Load: 0.00 MW, Number of DCs: 1
时间 00:00
     p_from_mw  p_to_mw  loading_percent
   -2000.0   2000.0            100.0
   -2000.0   2000.0            100.0
    2000.0  -2000.0            100.0
    2000.0  -2000.0            100.0
    2000.0  -2000.0            100.0
    总负荷: 90224.34000295299
    机组出力: 90224.34000329627
    总成本: 803791.0749335826
    绑定线路数: 10
    线路最高负载率: 100.00%
时间 01:00
     p_from_mw  p_to_mw  loading_percent
   -2000.0   2000.0            100.0
    2000.0  -2000.0            100.0
    2000.0  -2000.0            100.0
   -2000.0   2000.0            100.0
    2000.0  -2000.0            100.0
    总负荷: 83284.006156572
    机组出力: 83284.00615690465
    总成本: 593008.644168001
    绑定线路数: 10
    线路最高负载率: 100.00%
时间 02:00
       p_from_mw      p_to_mw  loading_percent
-2000.000000  2000.000000            100.0
 2000.000000 -2000.000000            100.0
-2000.000000  2000.000000            100.0
 1999.999997 -1999.999997            100.